In [0]:
# 1. Kaynak verimiz ZATEN Delta formatındaymış, bu yüzden format("delta") ile okuyoruz
delta_kaynak_yolu = "/Volumes/workspace/default/steam/delta/bronze/steam_reviews"
df_source = spark.read.format("delta").load(delta_kaynak_yolu)

# 2. Okuduğumuz Delta verisini Databricks üzerindeki kalıcı Bronze hedefimize kopyalıyoruz
delta_hedef_yolu = "/Volumes/workspace/default/steam/steam_reviews_bronze"
df_source.write.format("delta").mode("overwrite").save(delta_hedef_yolu)

print(f"Delta verisi başarıyla Databricks ana dizinine kopyalandı: {delta_hedef_yolu}")

# 3. EDA (Keşifçi Veri Analizi) adımı için veriyi okuyup kontrol edelim
df_delta = spark.read.format("delta").load(delta_hedef_yolu)

# Verinin ilk 5 satırını ve düzgün yüklenip yüklenmediğini görelim
display(df_delta.limit(5))

Delta verisi başarıyla Databricks ana dizinine kopyalandı: /Volumes/workspace/default/steam/steam_reviews_bronze


kafka_offset,kafka_partition,ingestion_time,timestamp,app_id,app_name,review_id,user_id,review_text,voted_up,votes_helpful,playtime_forever
3868187,0,2026-05-12T15:48:27.187Z,2026-05-12T15:43:49.016766,304050,Trove,rev_3868187,user_3868187,PAY TO WIN SIMPLE AS THAT,-1,0,0
3868188,0,2026-05-12T15:48:27.187Z,2026-05-12T15:43:49.017684,304050,Trove,rev_3868188,user_3868188,"It's crap! I downloaded it and it gave me a different gane called 'Glyph', I need an account for, I made one then it says that I don't have an account so want me recomendation? DON'T GET IT!",-1,0,0
3868189,0,2026-05-12T15:48:27.187Z,2026-05-12T15:43:49.018332,304050,Trove,rev_3868189,user_3868189,"Honestly, I at first loved the game. It was fun. Probably still is. But I am bored of it. Some of the Dungeons don't make sense whatsoever, the wait list to join a game can be from instantaneous, to 10 hours long. The multiplayer is tough, because you can defeat a dungeon, but then some ♥♥♥♥♥♥ comes in an steals all your stuff. It's not like you can attack them either. I don't like it anymore. But it might be fun. You download; and you decide.",-1,0,0
3868190,0,2026-05-12T15:48:27.187Z,2026-05-12T15:43:49.019284,304050,Trove,rev_3868190,user_3868190,"Its like terraria, but better and free.",1,0,0
3868191,0,2026-05-12T15:48:27.187Z,2026-05-12T15:43:49.020177,304050,Trove,rev_3868191,user_3868191,"don't mind all of the negative reviews, the game is still fun to play with friends and should not be taken seriously.",1,0,0


In [0]:
from pyspark.sql.functions import col

# 1. Bronze tablomuzu okuyalim
bronze_path = "/Volumes/workspace/default/steam/steam_reviews_bronze"
df_bronze = spark.read.format("delta").load(bronze_path)

print(f"Bronze Katmanı Satır Sayısı: {df_bronze.count()}")

# a) Duplike Kayitlari Kaldiralim (Kafka bazen aynı mesajı 2 kez gönderebilir)
df_silver = df_bronze.dropDuplicates()

# b) Kritik Sütunlardaki Bos Degerleri Temizleyelim
critical_columns = ["review_text", "voted_up", "app_id"]
df_silver = df_silver.dropna(subset=critical_columns)

print(f"Temizlenmis Silver Katmanı Satır Sayısı: {df_silver.count()}")

# 3. Kaydedelim
silver_path = "/Volumes/workspace/default/steam/steam_reviews_silver"
df_silver.write.format("delta").mode("overwrite").save(silver_path)

print("Silver katmani olusturuldu!")

Bronze Katmanı Satır Sayısı: 6417106
Temizlenmis Silver Katmanı Satır Sayısı: 6417106
Silver katmani olusturuldu!


In [0]:
from pyspark.sql.functions import col, count, avg, round, sum, when, lower

# 1. Temiz Silver tablomuzu okuyoruz
silver_path = "/Volumes/workspace/default/steam/steam_reviews_silver"
df_silver = spark.read.format("delta").load(silver_path)

# GÜVENLİ DÖNÜŞTÜRME: Kaggle verisinde olumlu=1, olumsuz=-1 olarak gelir.
voted_up_str = lower(col("voted_up").cast("string"))

safe_voted_up = when(voted_up_str.isin("1", "true"), 1) \
                .when(voted_up_str.isin("-1", "0", "false"), 0) \
                .otherwise(None)

# 2. Gold Katmani: Oyun bazında toplam yorum, olumlu yorum oranı ve toplam faydalı oy sayısı
df_gold = df_silver.groupBy("app_id", "app_name").agg(
    count("*").alias("toplam_yorum_sayisi"),
    round(avg(safe_voted_up) * 100, 2).alias("olumlu_yorum_yuzdesi"),
    sum(col("votes_helpful").cast("int")).alias("toplam_faydali_oy") # cast("int") ile güvene aldık
)

# 3. Gold tablosunu kaydediyoruz
gold_path = "/Volumes/workspace/default/steam/steam_reviews_gold_app_summary"
df_gold.write.format("delta").mode("overwrite").save(gold_path)

print("Gold katmani olusturuldu!")

display(df_gold.orderBy(col("toplam_yorum_sayisi").desc()).limit(10))

Gold katmani olusturuldu!


app_id,app_name,toplam_yorum_sayisi,olumlu_yorum_yuzdesi,toplam_faydali_oy
218620,PAYDAY 2,88973,69.42,14462
221100,DayZ,88850,66.23,14306
105600,Terraria,84828,97.08,8198
252490,Rust,77037,79.22,10310
570,Dota 2,73541,85.56,12509
252950,Rocket League,54227,94.27,6905
391540,Undertale,51918,96.02,7424
550,Left 4 Dead 2,50980,92.55,4455
230410,Warframe,48229,90.71,5522
271590,Grand Theft Auto V,42374,74.0,9538


In [0]:
# Yorumlarin (1 veya -1) veri seti icindeki dagilimini sayalim
score_distribution = df_silver.groupBy("voted_up").count().orderBy("voted_up")
print("Olumlu(1)/Olumsuz(-1) Yorum Dagilimi:")
display(score_distribution)

Olumlu(1)/Olumsuz(-1) Yorum Dagilimi:


voted_up,count
-1,1156686
1,5260420


In [0]:
from pyspark.sql.functions import col, sum, when

# 1. Bronze (Ham) veriyi okuyalım ki gerçek kayıpları görebilelim
bronze_path = "/Volumes/workspace/default/steam/steam_reviews_bronze"
df_bronze = spark.read.format("delta").load(bronze_path)

# 2. Her sütundaki eksik (Null) değer sayısını hesaplayalım
# String (Metin) tipindeki sütunlarda hata almamak için isnan() fonksiyonunu kaldırdık
eksik_degerler_df = df_bronze.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_bronze.columns
])

print("Bronze Katmanındaki Eksik Değerlerin Analizi:")
display(eksik_degerler_df)

Bronze Katmanındaki Eksik Değerlerin Analizi:


kafka_offset,kafka_partition,ingestion_time,timestamp,app_id,app_name,review_id,user_id,review_text,voted_up,votes_helpful,playtime_forever
0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
from pyspark.sql.functions import col

silver_path = "/Volumes/workspace/default/steam/steam_reviews_silver"
df_silver = spark.read.format("delta").load(silver_path)

# votes_helpful sütununu integer'a çevirerek sıralayalım ki metinsel ("9" > "10") sıralama hatası olmasın
df_silver_int = df_silver.withColumn("votes_helpful_int", col("votes_helpful").cast("int"))

populer_yorumlar = df_silver_int.orderBy(col("votes_helpful_int").desc()) \
                                .select("app_name", "voted_up", "votes_helpful_int", "review_text") \
                                .limit(10)

print("\nEn Çok Faydalı Bulunan (Popüler) 10 İnceleme:")
display(populer_yorumlar)


En Çok Faydalı Bulunan (Popüler) 10 İnceleme:


app_name,voted_up,votes_helpful_int,review_text
Batman™: Arkham Knight,1,1,"I've only played about 6 hours of the game, but I'm enjoying it with minimal frame rate issues, definitely none as bad as some others have described. The Bat Mobile is fun, but does handle really poorly, you can drift, but I haven't found it to be that useful. In fact it seemed like just releasing the gas worked better when making sharp turns. I feel like it would be better if Batman was able to take out the tanks without the use of the Batmobile because he's more resourceful than that, maybe la"
Batman™: Arkham Knight,1,1,"Recently switched from console to PC gaming, for the sole reason of getting this game on the best graphics. But being new to the PC gaming world so far I think the game looks great and plays great. I haven't had any issues that everyone else seems to be describing, but there has been quite a few updates that I had to install before I could play, so maybe that's the reason."
Batman™: Arkham Knight,-1,1,Too many glitches and bugs to play this post-patch on a GTX 970 with 8GB 1866mhz of ram. Torrent if you want to play it.
Batman™: Arkham Knight,1,1,Game works now.
Batman™: Arkham Knight,-1,1,Until this game gets fixed not even ganna look at the piece of junk
Batman™: Arkham Knight,-1,1,By the time the game was fixed... I moved on. I honestly don't care about this game anymore.
Batman™: Arkham Knight,1,1,"I can't say enough about this game! So freaking awesome!! Amazing graphics and framerates! I'm not sure what everyone is talking about, I can play this game maxxed out with all of the game works settings on and get almost 60fps (I turned the cap off) the whole time...it does drop when the batmobile is inbound and I'm diving off a sky skraper, but only to about 40-45 fps. Running on a 2nd gen i7, 8gb of ram, and a gtx 980. Freaking love this game! So much fun and worth the wait!!!"
Batman™: Arkham Knight,1,1,"It is a good game. I mean, this isn't amazing like I have expected, but its fine. The gameplay is nice, the fights are great, the story is good and the graphics are really great. The Batmobile controll isn't comfortable and even pretty hard for me to use. So I would recommend this game if you are looking for a nice game with good graphics."
Batman™: Arkham Knight,-1,1,This game has a nice main menu. unfortunately thats as far as i can get
Batman™: Arkham Knight,1,1,"Batman: Arkham Knight is a beautiful game, and it is full of content when you have the DLC's. The Good: 1) Beautiful 2) Full of content with the DLCs 3) Batman is back and tougher than ever. 4) The best interpretation of Batman through videogame, and argubly the best interpretation of Batman besides in the comics. 5) You can get a lot of hours out of it at a steady pace. The Bad: 1) You need all the DLCs to have a full content game 2) The story is kinda weak 3) *SPOILER* No boss fights reall"


In [0]:
# Temizlenmiş verimizi okuyalım
silver_path = "/Volumes/workspace/default/steam/steam_reviews_silver"
df_silver = spark.read.format("delta").load(silver_path)

# voted_up sütununun içinde TAM OLARAK hangi değerlerin olduğunu ve kaç defa geçtiklerini görelim
df_silver.groupBy("voted_up").count().orderBy(col("count").desc()).show(10, truncate=False)

+--------+-------+
|voted_up|count  |
+--------+-------+
|1       |5260420|
|-1      |1156686|
+--------+-------+

